<a href="https://colab.research.google.com/github/asaveraasad-data/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
import os
import subprocess

REPO_URL = "https://github.com/asaveraasad-data/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship/flyrank-ml-internship


In [25]:
import os

print(os.getcwd())
print(os.listdir())

/content/flyrank-ml-internship/flyrank-ml-internship
['README.md', 'work', 'notebooks', 'requirements.txt', '.github', 'skills', 'LICENSE', 'outputs', 'docs', 'submission', 'SETUP.md', 'scripts', '.git', 'data', 'DATA_USE.md', 'AGENTS.md', '.gitignore', 'CLAUDE.md', 'GUIDE.md']


## 1. Method choice and why

### Selected Method

I selected **Logistic Regression** as the primary model for this task.

My goal is to identify content pages that should be prioritized for content refresh. This is a binary classification problem because each page can either be classified as needing review or not needing review.

Logistic Regression is a good starting point because it is simple, interpretable, and provides prediction probabilities that can also be used for ranking pages. It serves as a transparent benchmark before considering more complex models such as Decision Trees or Random Forests.

The model will be compared against the Week 4 rule-based baseline using the same dataset, the same train/test split, and the same evaluation metrics.

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

RANDOM_STATE = 42

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Random seed:", RANDOM_STATE)
print("Dataset shape:", df.shape)

df.head()

Random seed: 42
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [27]:
# Create the target variable

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nPercentage:")
print(
    (df["is_declining_label"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Percentage:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


### Split Design

I used **GroupShuffleSplit** with `client_id` as the grouping variable.

This ensures that content from the same client never appears in both the training and testing sets. Grouped validation is more honest because it evaluates how well the model generalizes to completely unseen clients rather than memorizing client-specific patterns.

Training Set:
- 23,837 rows
- 25 unique clients

Testing Set:
- 6,163 rows
- 7 unique clients

Shared clients between train and test: **0**

In [28]:
# Honest train/test split grouped by client

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print()
print("Unique train clients:", train_df["client_id"].nunique())
print("Unique test clients:", test_df["client_id"].nunique())

print()
print(
    "Clients shared:",
    len(
        set(train_df["client_id"]).intersection(
            set(test_df["client_id"])
        )
    )
)

Train shape: (23837, 45)
Test shape: (6163, 45)

Unique train clients: 25
Unique test clients: 7

Clients shared: 0


### Model Performance

The Logistic Regression model was trained using five leakage-safe features:

- days_since_last_update
- impressions_90d
- ctr
- avg_position
- content_age_days

The same grouped train/test split was used for evaluation.

Results:

| Metric | Logistic Regression |
|---------|--------------------:|
| Accuracy | 0.5377 |
| Precision | 0.5433 |
| Recall | 0.5973 |
| F1 Score | 0.5691 |

This model provides an interpretable baseline that can later be compared with more complex models such as Decision Trees or Random Forests.

In [29]:
# Features used for the model

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

target = "is_declining_label"

X_train = train_df[features]
X_test = test_df[features]

y_train = train_df[target]
y_test = test_df[target]

print("Features:")
print(features)

print("\nMissing values:")
print(X_train.isnull().sum())

Features:
['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days']

Missing values:
days_since_last_update    0
impressions_90d           0
ctr                       0
avg_position              0
content_age_days          0
dtype: int64


### Error Analysis

The model correctly identifies many declining pages but also produces both false positives and false negatives.

The confusion matrix shows that the classes overlap considerably, indicating that predicting content decline from only five features is challenging.

Feature importance (based on Logistic Regression coefficients) shows that:

1. CTR has the strongest influence on predictions.
2. Days since last update also contributes positively.
3. Content age has a smaller effect.
4. Average position and impressions contribute very little.

This suggests that user engagement signals are more informative than raw visibility metrics for predicting declining content.

In [30]:
# Train Logistic Regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Model trained successfully!")

Model trained successfully!


**Evaluation**

In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

Accuracy : 0.5377
Precision: 0.5433
Recall   : 0.5973
F1 Score : 0.5691

Confusion Matrix
[[1433 1581]
 [1268 1881]]


In [32]:
# Create decline probabilities for the test pages

test_df = test_df.copy()

test_df["model_probability"] = model.predict_proba(X_test)[:, 1]

print(test_df[
    ["content_id", "model_probability", "is_declining_label"]
].head(10))

              content_id  model_probability  is_declining_label
0   content_304f48230142           0.554471                   1
1   content_a1fb4e703a9e           0.363260                   1
5   content_d4084a4bc775           0.594428                   1
13  content_a5a2fbc76336           0.642132                   0
19  content_af865035b328           0.541303                   1
20  content_0d748c484ab1           0.627254                   1
22  content_3fb46bec4413           0.515370                   1
23  content_2da6ae9d0882           0.328732                   1
25  content_033ae3e7aecf           0.574417                   1
26  content_72c5c2d73e5a           0.467379                   0


In [33]:
# Create the baseline score on the SAME test set

test_df["stale"] = (
    test_df["days_since_last_update"] >= 180
).astype(int)

test_df["visible"] = (
    test_df["impressions_90d"] >= 500
).astype(int)

# Use only training data to determine the CTR threshold
ctr_threshold = train_df["ctr"].median()

test_df["low_ctr"] = (
    test_df["ctr"] <= ctr_threshold
).astype(int)

test_df["baseline_score"] = (
    test_df["stale"] * 2
    + test_df["visible"] * 3
    + test_df["low_ctr"] * 2
)

print(test_df[
    [
        "content_id",
        "baseline_score",
        "model_probability",
        "is_declining_label"
    ]
].head(10))

              content_id  baseline_score  model_probability  \
0   content_304f48230142               3           0.554471   
1   content_a1fb4e703a9e               5           0.363260   
5   content_d4084a4bc775               5           0.594428   
13  content_a5a2fbc76336               2           0.642132   
19  content_af865035b328               0           0.541303   
20  content_0d748c484ab1               3           0.627254   
22  content_3fb46bec4413               3           0.515370   
23  content_2da6ae9d0882               0           0.328732   
25  content_033ae3e7aecf               2           0.574417   
26  content_72c5c2d73e5a               3           0.467379   

    is_declining_label  
0                    1  
1                    1  
5                    1  
13                   0  
19                   1  
20                   1  
22                   1  
23                   1  
25                   1  
26                   0  


In [34]:
# Compare how well each method ranks declining pages

def precision_at_k(df, score_column, k):
    ranked = df.sort_values(
        score_column,
        ascending=False
    ).head(k)

    return ranked["is_declining_label"].mean()


ks = [10, 25, 50, 100]

results = []

for k in ks:
    results.append({
        "K": k,
        "Baseline Precision": precision_at_k(
            test_df,
            "baseline_score",
            k
        ),
        "Model Precision": precision_at_k(
            test_df,
            "model_probability",
            k
        )
    })

results_df = pd.DataFrame(results)

print(results_df.round(4))

     K  Baseline Precision  Model Precision
0   10                0.70             0.70
1   25                0.76             0.60
2   50                0.66             0.54
3  100                0.61             0.54


In [35]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Scaled Logistic Regression
scaled_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ))
])

scaled_model.fit(X_train, y_train)

test_df["scaled_model_probability"] = (
    scaled_model.predict_proba(X_test)[:, 1]
)

print(
    test_df[
        [
            "content_id",
            "model_probability",
            "scaled_model_probability",
            "is_declining_label"
        ]
    ].head(10)
)

              content_id  model_probability  scaled_model_probability  \
0   content_304f48230142           0.554471                  0.554487   
1   content_a1fb4e703a9e           0.363260                  0.363272   
5   content_d4084a4bc775           0.594428                  0.594459   
13  content_a5a2fbc76336           0.642132                  0.642064   
19  content_af865035b328           0.541303                  0.541303   
20  content_0d748c484ab1           0.627254                  0.627279   
22  content_3fb46bec4413           0.515370                  0.515418   
23  content_2da6ae9d0882           0.328732                  0.328752   
25  content_033ae3e7aecf           0.574417                  0.574452   
26  content_72c5c2d73e5a           0.467379                  0.467384   

    is_declining_label  
0                    1  
1                    1  
5                    1  
13                   0  
19                   1  
20                   1  
22                   

In [36]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

test_df["rf_probability"] = (
    rf_model.predict_proba(X_test)[:, 1]
)

print(
    test_df[
        [
            "content_id",
            "rf_probability",
            "is_declining_label"
        ]
    ].head(10)
)

              content_id  rf_probability  is_declining_label
0   content_304f48230142        0.626126                   1
1   content_a1fb4e703a9e        0.496056                   1
5   content_d4084a4bc775        0.770959                   1
13  content_a5a2fbc76336        0.646687                   0
19  content_af865035b328        0.483154                   1
20  content_0d748c484ab1        0.651056                   1
22  content_3fb46bec4413        0.740361                   1
23  content_2da6ae9d0882        0.498355                   1
25  content_033ae3e7aecf        0.575420                   1
26  content_72c5c2d73e5a        0.652282                   0


In [37]:
# Compare Baseline vs Logistic Regression vs Random Forest

results = []

for k in [10, 25, 50, 100]:
    results.append({
        "K": k,
        "Baseline Precision": precision_at_k(
            test_df,
            "baseline_score",
            k
        ),
        "Logistic Regression": precision_at_k(
            test_df,
            "model_probability",
            k
        ),
        "Random Forest": precision_at_k(
            test_df,
            "rf_probability",
            k
        )
    })

comparison_df = pd.DataFrame(results)

print(comparison_df.round(4))

     K  Baseline Precision  Logistic Regression  Random Forest
0   10                0.70                 0.70           0.70
1   25                0.76                 0.60           0.84
2   50                0.66                 0.54           0.76
3  100                0.61                 0.54           0.73


In [38]:
from sklearn.metrics import average_precision_score

# PR-AUC for each model

baseline_pr_auc = average_precision_score(
    y_test,
    test_df["baseline_score"]
)

logistic_pr_auc = average_precision_score(
    y_test,
    test_df["model_probability"]
)

rf_pr_auc = average_precision_score(
    y_test,
    test_df["rf_probability"]
)

print("Baseline PR-AUC:", round(baseline_pr_auc, 4))
print("Logistic Regression PR-AUC:", round(logistic_pr_auc, 4))
print("Random Forest PR-AUC:", round(rf_pr_auc, 4))

Baseline PR-AUC: 0.5237
Logistic Regression PR-AUC: 0.5311
Random Forest PR-AUC: 0.6064


In [39]:
# Check the distribution of clients in the test set

print("Test clients:")
print(test_df["client_id"].nunique())

print("\nDeclining rate in test set:")
print(round(test_df["is_declining_label"].mean(), 4))

print("\nDeclining rate by client:")
print(
    test_df.groupby("client_id")["is_declining_label"]
    .mean()
    .round(3)
)

Test clients:
7

Declining rate in test set:
0.511

Declining rate by client:
client_id
client_434c9b5ae5    0.517
client_4e07408562    0.493
client_8527a891e2    0.437
client_8b940be7fb    0.321
client_bdd2d3af3a    0.750
client_e629fa6598    0.456
client_f369cb89fc    0.601
Name: is_declining_label, dtype: float64


In [40]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

target = "is_declining_label"

robustness_results = []

splitter = GroupShuffleSplit(
    n_splits=5,
    test_size=0.2,
    random_state=42
)

for split_number, (train_idx, test_idx) in enumerate(
    splitter.split(df, groups=df["client_id"]),
    start=1
):

    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    X_train = train[features]
    X_test = test[features]
    y_train = train[target]
    y_test = test[target]

    # Train Random Forest
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    test["rf_probability"] = rf.predict_proba(X_test)[:, 1]

    # Baseline threshold learned only from training data
    ctr_threshold = train["ctr"].median()

    test["baseline_score"] = (
        (test["days_since_last_update"] >= 180).astype(int) * 2
        + (test["impressions_90d"] >= 500).astype(int) * 3
        + (test["ctr"] <= ctr_threshold).astype(int) * 2
    )

    row = {
        "split": split_number,
        "rf_pr_auc": average_precision_score(
            y_test,
            test["rf_probability"]
        )
    }

    for k in [25, 50, 100]:
        row[f"rf_precision_at_{k}"] = (
            test.sort_values(
                "rf_probability",
                ascending=False
            )
            .head(k)["is_declining_label"]
            .mean()
        )

        row[f"baseline_precision_at_{k}"] = (
            test.sort_values(
                "baseline_score",
                ascending=False
            )
            .head(k)["is_declining_label"]
            .mean()
        )

    robustness_results.append(row)

robustness_df = pd.DataFrame(robustness_results)

print(robustness_df.round(4))

   split  rf_pr_auc  rf_precision_at_25  baseline_precision_at_25  \
0      1     0.6064                0.84                      0.76   
1      2     0.8283                1.00                      0.76   
2      3     0.6156                0.48                      0.80   
3      4     0.5895                0.60                      0.32   
4      5     0.5104                0.68                      0.28   

   rf_precision_at_50  baseline_precision_at_50  rf_precision_at_100  \
0                0.76                      0.66                 0.73   
1                1.00                      0.78                 0.97   
2                0.46                      0.76                 0.55   
3                0.66                      0.36                 0.69   
4                0.60                      0.30                 0.59   

   baseline_precision_at_100  
0                       0.61  
1                       0.76  
2                       0.79  
3                       0.36

In [41]:
# Random Forest feature importance

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print(importance_df.round(4))

                  Feature  Importance
1         impressions_90d      0.3452
4        content_age_days      0.2673
3            avg_position      0.2466
2                     ctr      0.0775
0  days_since_last_update      0.0635


In [42]:
# Create the final ranked content recommendation queue

final_queue = test_df.copy()

final_queue["final_score"] = final_queue["rf_probability"]

# Rank highest probability first
final_queue = (
    final_queue
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)

final_queue["final_rank"] = (
    final_queue.index + 1
)

# Assign an action
final_queue["suggested_action"] = np.where(
    final_queue["final_score"] >= 0.70,
    "Prioritize for Content Review",
    np.where(
        final_queue["final_score"] >= 0.50,
        "Review / Monitor",
        "Monitor"
    )
)

print(
    final_queue[
        [
            "final_rank",
            "content_id",
            "final_score",
            "suggested_action",
            "is_declining_label",
            "impressions_90d",
            "content_age_days",
            "avg_position",
            "ctr",
            "days_since_last_update"
        ]
    ].head(25)
)

    final_rank            content_id  final_score  \
0            1  content_9b28cf5ae4f3     0.817382   
1            2  content_643b51c0a848     0.805755   
2            3  content_fb79d33c4d8a     0.796973   
3            4  content_d38581402cd5     0.796848   
4            5  content_91c02d2830b8     0.796666   
5            6  content_48cc6894c726     0.796347   
6            7  content_73ee00dad7f8     0.796339   
7            8  content_b08562686d22     0.796201   
8            9  content_67565df5500c     0.796182   
9           10  content_ecef382bc4ff     0.796012   
10          11  content_f8b232498008     0.795831   
11          12  content_452a4e18212c     0.795517   
12          13  content_72f65ab80b1d     0.795384   
13          14  content_2a228ce7aa1b     0.795094   
14          15  content_26d48a980581     0.794574   
15          16  content_0c8cb6bad4a7     0.793576   
16          17  content_9e8671965fff     0.793470   
17          18  content_d81093249cd6     0.793

In [43]:
# Create interpretable reason codes for the final recommendation queue

def get_reason_codes(row):
    reasons = []

    if row["impressions_90d"] >= 500:
        reasons.append("high impressions")

    if row["content_age_days"] >= 365:
        reasons.append("older content")

    if row["avg_position"] >= 10:
        reasons.append("weaker search position")

    if row["ctr"] <= df["ctr"].median():
        reasons.append("lower CTR")

    if row["days_since_last_update"] >= 180:
        reasons.append("not recently updated")

    if not reasons:
        reasons.append("no strong single signal")

    return "; ".join(reasons)

In [44]:
# Train final Random Forest on the full dataset
# This model is used only to generate the final recommendation queue.

from sklearn.ensemble import RandomForestClassifier

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

target = "is_declining_label"

X_full = df[features]
y_full = df[target]

final_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full, y_full)

# Score every page
df["model_probability"] = final_model.predict_proba(X_full)[:, 1]

print("Final model trained.")
print("Pages scored:", len(df))

Final model trained.
Pages scored: 30000


In [45]:
# Build final ranked recommendation queue

recommendations = (
    df.copy()
    .sort_values("model_probability", ascending=False)
    .reset_index(drop=True)
)

recommendations["final_rank"] = recommendations.index + 1

recommendations["suggested_action"] = recommendations["final_rank"].apply(
    lambda rank:
        "Prioritize for Content Review"
        if rank <= 25
        else "Review / Monitor"
        if rank <= 100
        else "Monitor"
)

recommendations["reason_codes"] = recommendations.apply(
    get_reason_codes,
    axis=1
)

final_output = recommendations[
    [
        "final_rank",
        "content_id",
        "model_probability",
        "suggested_action",
        "reason_codes",
        "impressions_90d",
        "content_age_days",
        "avg_position",
        "ctr",
        "days_since_last_update"
    ]
]

print(final_output.head(25))

    final_rank            content_id  model_probability  \
0            1  content_3f8c7aed9d6c           0.814746   
1            2  content_6dd02d2e1d19           0.813490   
2            3  content_4c2aae7232ac           0.810091   
3            4  content_5cda32af644c           0.809996   
4            5  content_2744dcacbe42           0.809697   
5            6  content_7d5ad3f9feee           0.809685   
6            7  content_579414680ff1           0.809658   
7            8  content_02bcf3eec147           0.809218   
8            9  content_4bb993e9270e           0.809107   
9           10  content_1c0f8c2f9018           0.808967   
10          11  content_b5964a1a276f           0.808646   
11          12  content_3cd91a252207           0.808315   
12          13  content_d0fd2e1ec8f6           0.808198   
13          14  content_56c54a03ece2           0.807632   
14          15  content_ffeed86ef360           0.807572   
15          16  content_7ae230422776           0.807064 

In [46]:
# Save final ranked recommendation queue

os.makedirs("work/outputs", exist_ok=True)

final_output.to_csv(
    "work/outputs/final_content_recommendations.csv",
    index=False
)

print("Saved:")
print("work/outputs/final_content_recommendations.csv")

print("\nRows:", len(final_output))
print("\nAction distribution:")
print(final_output["suggested_action"].value_counts())

Saved:
work/outputs/final_content_recommendations.csv

Rows: 30000

Action distribution:
suggested_action
Monitor                          29900
Review / Monitor                    75
Prioritize for Content Review       25
Name: count, dtype: int64


In [47]:
# Save model feature importance for the paper

final_importance = pd.DataFrame({
    "feature": features,
    "importance": final_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

final_importance.to_csv(
    "work/outputs/model_feature_importance.csv",
    index=False
)

print(final_importance.round(4))

                  feature  importance
1         impressions_90d      0.3676
4        content_age_days      0.2523
3            avg_position      0.2412
2                     ctr      0.0741
0  days_since_last_update      0.0648


**Feature Interpretation**

In [48]:
# Logistic Regression coefficients

coef = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
})

coef["Absolute"] = coef["Coefficient"].abs()
coef = coef.sort_values("Absolute", ascending=False)

coef

,Feature,Coefficient,Absolute
2,ctr,-0.055671,0.055671
0,days_since_last_update,0.005664,0.005664
4,content_age_days,-0.003083,0.003083
3,avg_position,-0.000163,0.000163
1,impressions_90d,-0.000004,0.000004


## Self-check

Before you submit, confirm each line honestly:

✅Every section above is filled — markdown thinking AND the code that backs it

✅The notebook runs top to bottom with no errors (Runtime → Run all)

✅No client names, URLs, or private queries anywhere

✅My claims use careful words: observed, measured, directional, decision-support

✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.